# 02 · Pelatihan U-Net dan DeepLabV3+ (T4-A, Raabin-WBC)

**Cara pakai (tiap anggota, di akun Colab masing-masing)**
1. Ketua sudah menaruh `data.py`, `runlog.py` (dan folder `splits/`) di folder Drive bersama. Taruh `metrics.py`, `models.py`, `train.py` di folder `code/` yang sama.
2. Runtime → Change runtime type → **T4 GPU**.
3. Isi `OWNER` di sel setup dengan **satu nama tetap** (mis. nama depan). Nama ini menentukan file log `logs/log_<OWNER>.csv`.
4. Jalankan sel setup, lalu **sel uji cepat**. Setelah itu jalankan hanya sel milikmu.

**Aturan**
- Satu faktor berubah per perbandingan, sisanya tetap pada baseline (U-Net ResNet-34, CE+Dice, split `grouped`, augmentasi warna aktif, 256 px, seed 42).
- Semua pemilihan epoch memakai **validasi**. `eval_test=True` hanya untuk konfigurasi final, sekali, di akhir.
- Jangan mengubah file di `splits/`.
- Run yang gagal tetap tercatat otomatis. Run yang terputus karena sesi Colab habis TIDAK tercatat, catat manual.

In [ ]:
# ===== SEL SETUP (jalankan tiap sesi Colab baru) =====
from google.colab import drive; drive.mount('/content/drive')
!apt-get install -y unrar > /dev/null 2>&1
!pip -q install albumentations segmentation-models-pytorch
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"

!mkdir -p /content/raabin_gt/bmp
!unrar x -inul "{DRIVE}/GroundTruths_bmpformat.rar" /content/raabin_gt/bmp/

import sys; sys.path.append(f"{DRIVE}/code")
import warnings; warnings.filterwarnings("ignore")
from train import run_experiment, run_seeds

OWNER = "Based"      # satu nama tetap, tanpa spasi
assert OWNER and " " not in OWNER, "OWNER tidak boleh kosong atau mengandung spasi"
print("OWNER =", OWNER)

Mounted at /content/drive
OWNER = Based


## Uji cepat (1 epoch)
Memastikan GPU, unduhan bobot pralatih, dan penulisan log berjalan. Tercatat di log dengan `ablation='smoke_test'`; baris ini dibuang saat laporan. Perkiraan waktu 1-3 menit (belum diukur di T4).

In [ ]:
run_experiment(OWNER, "smoke_test", "-", epochs=1)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/1 | loss 0.6967 | val dice cyto 0.8143 nuc 0.9607 | skor 0.8875


{'val_dice_bg': 0.9873,
 'val_iou_bg': 0.9753,
 'val_dice_cyto': 0.8143,
 'val_dice_cyto_sd': 0.1076,
 'val_iou_cyto': 0.6991,
 'val_dice_nuc': 0.9607,
 'val_dice_nuc_sd': 0.0289,
 'val_iou_nuc': 0.9257,
 'val_miou': 0.8667,
 'val_miou_sd': 0.0482,
 'val_hd95_cyto': 21.4363,
 'val_hd95_cyto_sd': 18.2784,
 'val_hd95_nuc': 7.3815,
 'val_hd95_nuc_sd': 9.709,
 'run_id': 'Based_1123_070804',
 'best_epoch': 1}

## Baseline (dijalankan SEKALI untuk seluruh tim)
Baseline U-Net milik **Anggota 2**, baseline DeepLabV3+ milik **Anggota 3**. Semua ablasi dibandingkan dengan baseline U-Net ini. Jangan dijalankan ulang oleh anggota lain.

In [ ]:
# Baseline U-Net (Anggota 2)
run_experiment(OWNER, "baseline", "unet_ce_dice", arch="unet")

In [ ]:
# Baseline DeepLabV3+ (Anggota 3)
run_experiment(OWNER, "baseline", "deeplabv3plus_ce_dice", arch="deeplabv3plus")

## Ablasi per anggota (U-Net, 1 seed = 42, evaluasi di VALIDASI)
Nilai yang sama dengan baseline tidak dijalankan lagi. Jika memori GPU habis (terutama 384 px), turunkan `batch_size` ke 8 dan itu tercatat di log.

In [ ]:
# Anggota 1 (Ketua): skema split. Baseline memakai 'grouped'; ini menambah 'random' (bocor antar-apusan)
run_experiment(OWNER, "split", "random", split="random")

ep  1/20 | loss 0.6958 | val dice cyto 0.8333 nuc 0.9066 | skor 0.8700
ep  2/20 | loss 0.2136 | val dice cyto 0.8645 nuc 0.9501 | skor 0.9073
ep  3/20 | loss 0.1661 | val dice cyto 0.8893 nuc 0.9608 | skor 0.9250
ep  4/20 | loss 0.1518 | val dice cyto 0.8885 nuc 0.9569 | skor 0.9227
ep  5/20 | loss 0.1437 | val dice cyto 0.8954 nuc 0.9652 | skor 0.9303
ep  6/20 | loss 0.1311 | val dice cyto 0.8922 nuc 0.9430 | skor 0.9176
ep  7/20 | loss 0.1252 | val dice cyto 0.8578 nuc 0.9354 | skor 0.8966
ep  8/20 | loss 0.1263 | val dice cyto 0.9056 nuc 0.9659 | skor 0.9357
ep  9/20 | loss 0.1150 | val dice cyto 0.8961 nuc 0.9575 | skor 0.9268
ep 10/20 | loss 0.1210 | val dice cyto 0.8951 nuc 0.9561 | skor 0.9256
ep 11/20 | loss 0.1130 | val dice cyto 0.9084 nuc 0.9625 | skor 0.9355
ep 12/20 | loss 0.1137 | val dice cyto 0.9094 nuc 0.9658 | skor 0.9376
ep 13/20 | loss 0.1038 | val dice cyto 0.9115 nuc 0.9695 | skor 0.9405
ep 14/20 | loss 0.1042 | val dice cyto 0.9134 nuc 0.9673 | skor 0.9404
ep 15/

{'val_dice_bg': 0.9938,
 'val_iou_bg': 0.9877,
 'val_dice_cyto': 0.9183,
 'val_dice_cyto_sd': 0.0471,
 'val_iou_cyto': 0.8521,
 'val_dice_nuc': 0.9699,
 'val_dice_nuc_sd': 0.0128,
 'val_iou_nuc': 0.9419,
 'val_miou': 0.9272,
 'val_miou_sd': 0.0277,
 'val_hd95_cyto': 8.98,
 'val_hd95_cyto_sd': 9.5948,
 'val_hd95_nuc': 5.0535,
 'val_hd95_nuc_sd': 9.3549,
 'run_id': 'Based_1123_070844',
 'best_epoch': 20}

In [ ]:
# Anggota 2: fungsi loss. Baseline = ce_dice; tambahkan tiga lainnya
for l in ["ce", "dice", "focal_dice"]:
    run_experiment(OWNER, "loss", l, loss=l)

In [ ]:
# Anggota 2
run_experiment(OWNER, "loss", "ce_p20", loss="ce", patience=20)

In [ ]:
# Anggota 3: augmentasi warna. Baseline = aktif; ini menambah 'nonaktif'
run_experiment(OWNER, "color_aug", "off", color_aug=False)

In [ ]:
# Anggota 3
run_experiment(OWNER, "color_aug", "off_p20", color_aug=False, patience=20)

In [ ]:
# Anggota 4: resolusi masukan. Baseline = 256; ini menambah 384
run_experiment(OWNER, "resolution", "384", img_size=384)

In [ ]:
# Opsional bila waktu cukup: encoder pralatih vs inisialisasi acak
run_experiment(OWNER, "pretrained", "random_init", pretrained=False)

ep  1/20 | loss 0.7325 | val dice cyto 0.5574 nuc 0.8683 | skor 0.7128
ep  2/20 | loss 0.2991 | val dice cyto 0.7696 nuc 0.9604 | skor 0.8650
ep  3/20 | loss 0.2584 | val dice cyto 0.7555 nuc 0.9365 | skor 0.8460
ep  4/20 | loss 0.2177 | val dice cyto 0.7778 nuc 0.9659 | skor 0.8719
ep  5/20 | loss 0.2002 | val dice cyto 0.7228 nuc 0.9158 | skor 0.8193
ep  6/20 | loss 0.1827 | val dice cyto 0.8269 nuc 0.9544 | skor 0.8906
ep  7/20 | loss 0.1789 | val dice cyto 0.8407 nuc 0.9618 | skor 0.9012
ep  8/20 | loss 0.1698 | val dice cyto 0.8315 nuc 0.9636 | skor 0.8976
ep  9/20 | loss 0.1598 | val dice cyto 0.8467 nuc 0.9576 | skor 0.9022
ep 10/20 | loss 0.1482 | val dice cyto 0.8568 nuc 0.9678 | skor 0.9123
ep 11/20 | loss 0.1448 | val dice cyto 0.8577 nuc 0.9662 | skor 0.9120
ep 12/20 | loss 0.1467 | val dice cyto 0.8428 nuc 0.9581 | skor 0.9004
ep 13/20 | loss 0.1349 | val dice cyto 0.8710 nuc 0.9685 | skor 0.9198
ep 14/20 | loss 0.1289 | val dice cyto 0.8644 nuc 0.9647 | skor 0.9145
ep 15/

{'val_dice_bg': 0.993,
 'val_iou_bg': 0.9862,
 'val_dice_cyto': 0.879,
 'val_dice_cyto_sd': 0.0945,
 'val_iou_cyto': 0.7947,
 'val_dice_nuc': 0.971,
 'val_dice_nuc_sd': 0.0122,
 'val_iou_nuc': 0.9439,
 'val_miou': 0.9083,
 'val_miou_sd': 0.0443,
 'val_hd95_cyto': 11.7717,
 'val_hd95_cyto_sd': 18.7364,
 'val_hd95_nuc': 4.0816,
 'val_hd95_nuc_sd': 1.2222,
 'run_id': 'Based_1123_071214',
 'best_epoch': 18}

## Perbandingan utama: 3 seed untuk dua konfigurasi terbaik
Pilih dua konfigurasi berdasarkan hasil **validasi** (misalnya U-Net vs DeepLabV3+ dengan pengaturan terbaik). `eval_test=True` menyentuh set uji, jadi jalankan ini **hanya setelah semua ablasi selesai dan konfigurasinya final**, sekali saja. `save_path` menyimpan bobot untuk model card (satu file ~100 MB).

In [ ]:
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "unet_final", arch="unet", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/unet_final_seed{s}.pt"))
df_unet = pd.DataFrame(res)
cols = [c for c in df_unet.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_unet[cols].mean(), "sd": df_unet[cols].std(ddof=1)}).round(4))

In [ ]:
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "deeplabv3plus_final", arch="deeplabv3plus", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/deeplabv3plus_final_seed{s}.pt"))
df_dlv3 = pd.DataFrame(res)
cols = [c for c in df_dlv3.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_dlv3[cols].mean(), "sd": df_dlv3[cols].std(ddof=1)}).round(4))

## Gabung seluruh log (Ketua, di akhir)
Menghasilkan tabel log eksperimen, total jam GPU, dan bukti bahwa set uji hanya dipakai pada run final.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, pandas as pd
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"

files = sorted(glob.glob(f"{DRIVE}/logs/log_*.csv"))
print("File log:", [f.split("/")[-1] for f in files])
log = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
log = log[log.ablation != "smoke_test"]
log.to_csv(f"{DRIVE}/logs/LOG_GABUNGAN.csv", index=False)

print("\nTotal run:", len(log), "| gagal:", (log.status == "failed").sum())
print("Total jam GPU:", round(log.gpu_minutes.sum() / 60, 2))
print("Run yang memakai set uji:", int(log.used_test.sum()))
print(log.groupby("owner").gpu_minutes.sum().div(60).round(2))
print("\nRincian per ablasi:")
print(log.groupby(["ablation", "factor_value"]).size())
print("\nRun main (final):")
print(log[log.ablation == "main"][["owner", "arch", "seed", "best_epoch", "epochs_run"]].to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File log: ['log_Adit (1).csv', 'log_Adit (2).csv', 'log_Adit (3).csv', 'log_Adit (4).csv', 'log_Adit (5).csv', 'log_Adit (6).csv', 'log_Adit (7).csv', 'log_Adit (8).csv', 'log_Adit.csv', 'log_Alpin (1).csv', 'log_Alpin.csv', 'log_Based.csv', 'log_Iklil (1).csv', 'log_Iklil (2).csv', 'log_Iklil (3).csv', 'log_Iklil (4).csv', 'log_Iklil (5).csv', 'log_Iklil (6).csv', 'log_Iklil.csv']

Total run: 17 | gagal: 0
Total jam GPU: 0.8
Run yang memakai set uji: 6
owner
Adit     0.36
Alpin    0.08
Based    0.10
Iklil    0.26
Name: gpu_minutes, dtype: float64

Rincian per ablasi:
ablation    factor_value         
baseline    deeplabv3plus_ce_dice    1
            unet_ce_dice             1
color_aug   off                      1
            off_p20                  1
loss        ce                       1
            ce_p20                   1
            dice            

In [ ]:
import glob, pandas as pd
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"
raw = pd.concat([pd.read_csv(f) for f in glob.glob(f"{DRIVE}/logs/log_*.csv")], ignore_index=True)
print("Baris mentah:", len(raw), "| run_id ganda:", raw.run_id.duplicated().sum())
print("Jam GPU termasuk smoke_test:", round(raw.gpu_minutes.sum() / 60, 2))
print("Smoke test:", int((raw.ablation == "smoke_test").sum()), "baris")

Baris mentah: 26 | run_id ganda: 0
Jam GPU termasuk smoke_test: 0.88
Smoke test: 9 baris


In [1]:
!pip freeze | grep -i -E "^(torch|segmentation|albumentations|scipy|pandas|numpy|scikit-learn|matplotlib|pillow|imagehash)=="

albumentations==2.0.8
matplotlib==3.10.0
numpy==2.4.6
pandas==2.2.3
pillow==11.3.0
scikit-learn==1.6.1
scipy==1.16.3
torch==2.14.1+cu130


In [2]:
!pip freeze | grep -i -E "^(segmentation[-_]models[-_]pytorch|imagehash|timm|torchvision)=="
!python --version

timm==1.0.31
torchvision==0.29.1+cu130
Python 3.13.16


In [3]:
!pip -q install segmentation-models-pytorch imagehash
!pip freeze | grep -i -E "^(segmentation[-_]models[-_]pytorch|imagehash)=="

ImageHash==4.3.2
segmentation_models_pytorch==0.6.0
